# Solutions · Chapter 05-04 · Metrics

Worked answers to every exercise in `notebooks/05_regression/05-04_metrics.ipynb`.

Read the exercise, attempt it, and only then read on. A solution you have not first struggled with
teaches almost nothing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

# SYNTHETIC: 05-02's nine deliveries, and the chapter's predictions for them.
actual = np.array([16, 20, 25, 25, 30, 31, 36, 42, 45], dtype=float)
predicted = np.array([16, 19.5, 23, 26.5, 30, 33.5, 37, 40.5, 44], dtype=float)


def mean_absolute(a, p):
    return float(np.abs(a - p).mean())


def mean_squared(a, p):
    return float(((a - p) ** 2).mean())


def root_mean_squared(a, p):
    return float(np.sqrt(mean_squared(a, p)))


def percentage(a, p):
    return float(np.mean(np.abs(a - p) / np.abs(a)))


def symmetric_percentage(a, p):
    return float(np.mean(2 * np.abs(a - p) / (np.abs(a) + np.abs(p))))


print("MAE %.4f   RMSE %.4f" % (mean_absolute(actual, predicted),
                                root_mean_squared(actual, predicted)))

## Quick understanding

### E1 · Units

| Metric | Units |
|---|---|
| MAE | minutes |
| MSE | minutes **squared** |
| RMSE | minutes |
| MAPE | a fraction, or a percentage - unitless |
| R-squared | unitless |

The one that catches people is MSE. "Our error is 1.89" invites the reader to hear 1.89 minutes, and the
true statement is 1.89 minutes-squared, which nobody can picture. That is the whole argument for taking
the square root before reporting.

### E2 · Why RMSE is never smaller than MAE

Squaring is a convex function, so the mean of the squares is at least the square of the mean
(Jensen's inequality): `mean(e²) >= mean(|e|)²`. Taking square roots gives `RMSE >= MAE`.

Concretely: squaring stretches large numbers more than small ones, so the mean square is pulled towards
the biggest error while the mean absolute value is not.

**A ratio of exactly 1 means every error has the same size.** That is the only way the two agree, and in
practice it means either a very artificial situation or a bug - real residuals vary.

### E3 · MAPE's three failures

1. **Asymmetric** - over-prediction can cost any amount, under-prediction is capped at 100%, so a model
   tuned on MAPE is pushed towards predicting low.
2. **Explodes near zero** - dividing by an actual close to zero makes one row dominate the average, and
   an actual of exactly zero makes it infinite.
3. **Not comparable across series** - "MAPE 5%" means different absolute errors on different scales, so
   two teams' MAPEs cannot be added, averaged or ranked against each other.

## Hand calculation

### E4 · Errors 1, 1, 1, 5

- **MAE** = (1 + 1 + 1 + 5) / 4 = 8 / 4 = **2**
- **MSE** = (1 + 1 + 1 + 25) / 4 = 28 / 4 = **7**
- **RMSE** = sqrt(7) = **2.6458**
- **ratio** = 2.6458 / 2 = **1.3229**

### E5 · Replace the 5 with a 1

All four errors are now 1, so MAE = 1, MSE = 1, RMSE = 1, ratio = 1.

| | before | after | shrank by a factor of |
|---|---|---|---|
| MAE | 2 | 1 | 2.00 |
| MSE | 7 | 1 | 7.00 |
| RMSE | 2.6458 | 1 | 2.65 |

**MSE moved most, by a factor of 7.** One row out of four carried three quarters of the MSE, and removing
it removed six sevenths of the number. That is the outlier sensitivity of the chapter's headline example
in four rows you can do on paper.

In [ ]:
errors_before = np.array([1.0, 1.0, 1.0, 5.0])
errors_after = np.array([1.0, 1.0, 1.0, 1.0])

for name, errors in [("1, 1, 1, 5", errors_before), ("1, 1, 1, 1", errors_after)]:
    mae = float(np.abs(errors).mean())
    mse = float((errors ** 2).mean())
    print("%-10s   MAE %.4f   MSE %.4f   RMSE %.4f   ratio %.4f"
          % (name, mae, mse, np.sqrt(mse), np.sqrt(mse) / mae))

### E6 · Actual 80 predicted 100, then actual 100 predicted 80

**Case A - actual 80, predicted 100.** The absolute error is 20.
MAPE = 20 / 80 = **25%**.  sMAPE = 2 x 20 / (80 + 100) = 40 / 180 = **22.22%**.

**Case B - actual 100, predicted 80.** The absolute error is again 20.
MAPE = 20 / 100 = **20%**.  sMAPE = 2 x 20 / (100 + 80) = 40 / 180 = **22.22%**.

**sMAPE gives the same answer both times; MAPE does not.** MAPE divides by the actual alone, so the same
20-minute miss is worth more when the actual is small. sMAPE divides by the average of the two, and that
average is the same number in both directions - which is exactly what "symmetric" is naming.

Note that the two cases are not a fair test of MAPE's *bias*: they swap the roles of the two numbers
rather than over- and under-predicting the same actual. The chapter's ratio plot is the honest version of
that question.

In [ ]:
for description, a, p in [("actual 80, predicted 100", 80.0, 100.0),
                          ("actual 100, predicted 80", 100.0, 80.0)]:
    print("%-26s  MAPE %5.2f%%   sMAPE %5.2f%%"
          % (description, 100 * abs(a - p) / a, 100 * 2 * abs(a - p) / (a + p)))

### E7 · MSE 400 euros-squared

**RMSE = sqrt(400) = 20 euros.**

Report the RMSE. "Our model is typically about 20 euros out" is a sentence someone can act on; "our error
is 400 euros-squared" is not a quantity anyone has intuition for, and a reader who skims it will hear
"400 euros" and think the model is twenty times worse than it is.

Better still, report **MAE alongside it**. RMSE 20 tells you the squared-error size; whether that comes
from every row being 20 out or from one row being 200 out is the question the MAE answers.

## Coding

### E8 · A reporting function

In [ ]:
def report(a, p, baseline):
    mae = mean_absolute(a, p)
    rmse = root_mean_squared(a, p)
    baseline_mae = mean_absolute(a, baseline)
    print("MAE           %8.4f" % mae)
    print("RMSE          %8.4f" % rmse)
    print("RMSE / MAE    %8.4f" % (rmse / mae))
    print("baseline MAE  %8.4f" % baseline_mae)
    print("skill         %8.4f   (1 - MAE / baseline MAE)" % (1 - mae / baseline_mae))


median_constant = float(np.median(actual))
print("the median constant is %.1f minutes" % median_constant)
report(actual, predicted, np.full(actual.shape, median_constant))

**Skill 0.8529** - the model removes 85% of the error the best constant leaves. That single line is the
form every later chapter in this module reports in: a number in the target's units, and what it beat.

The ratio 1.24 is the free extra: errors here vary, but no single delivery is carrying the score.

### E9 · What ratio do normal errors produce?

In [ ]:
def ratio_of(errors):
    return float(np.sqrt((errors ** 2).mean()) / np.abs(errors).mean())


sim_rng = np.random.default_rng(0)
normal_ratios = np.array([ratio_of(e) for e in sim_rng.normal(0, 1, (2000, 50))])
heavy_ratios = np.array([ratio_of(e) for e in sim_rng.standard_t(3, (2000, 50))])

print("normal errors      mean ratio %.4f   90th percentile %.4f"
      % (normal_ratios.mean(), np.percentile(normal_ratios, 90)))
print("heavy-tailed (t3)  mean ratio %.4f   90th percentile %.4f"
      % (heavy_ratios.mean(), np.percentile(heavy_ratios, 90)))
print("\nsqrt(pi / 2) = %.4f" % np.sqrt(np.pi / 2))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.2))
bins = np.linspace(1.0, 2.6, 70)
ax.hist(normal_ratios, bins=bins, color="#0072B2", alpha=0.75, label="normal errors")
ax.hist(heavy_ratios, bins=bins, color="#D55E00", alpha=0.65, label="heavy-tailed errors (t, 3 df)")
ax.axvline(np.sqrt(np.pi / 2), color="#000000", linestyle="--", linewidth=1.8,
           label="sqrt(pi/2) = 1.2533")
ax.set_xlabel("RMSE / MAE on 50 errors")
ax.set_ylabel("how many of the 2,000 simulations")
ax.set_title("The ratio is a distribution shape detector", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Normal errors cluster at 1.2491, essentially sqrt(pi/2) = 1.2533.** That is not a coincidence: for a
normal distribution the mean absolute value is `sigma * sqrt(2/pi)` and the root mean square is `sigma`,
so the ratio is `sqrt(pi/2)` and the sigma cancels. **The value does not depend on the error scale at
all** - which is what makes the ratio a usable diagnostic rather than another thing to calibrate.

**Heavy-tailed errors average 1.4552, and their 90th percentile reaches 1.66** against 1.31 for the
normal. The two histograms barely overlap in the upper tail.

So the practical reading is: **around 1.25, unremarkable. Meaningfully above it, go and look at the
largest residuals** - something is producing errors far bigger than the typical one.

### E10 · The MAPE-optimal constant

In [ ]:
grid = np.linspace(10, 50, 4001)
mape_of_constant = np.array([percentage(actual, np.full(actual.shape, c)) for c in grid])
best_constant = float(grid[int(np.argmin(mape_of_constant))])

print("constant minimising MAPE : %.2f   (MAPE %.4f)" % (best_constant, mape_of_constant.min()))
print("the median               : %.2f   (MAPE %.4f)"
      % (np.median(actual), percentage(actual, np.full(actual.shape, np.median(actual)))))
print("the mean                 : %.2f   (MAPE %.4f)"
      % (actual.mean(), percentage(actual, np.full(actual.shape, actual.mean()))))
print()
print("but scored with MAE instead:")
print("  the MAPE-optimal constant : %.4f" % mean_absolute(actual, np.full(actual.shape, best_constant)))
print("  the median                : %.4f" % mean_absolute(actual, np.full(actual.shape, 30.0)))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(grid, 100 * mape_of_constant, color="#7B3294", linewidth=2.2)
ax.plot([best_constant], [100 * mape_of_constant.min()], "o", color="#7B3294", markersize=10)
ax.axvline(np.median(actual), color="#0072B2", linestyle="--", linewidth=1.8,
           label="the median, %.0f" % np.median(actual))
ax.axvline(best_constant, color="#7B3294", linestyle=":", linewidth=1.8,
           label="MAPE's choice, %.0f" % best_constant)
for value in actual:
    ax.plot([value], [1.5], "|", color="#666666", markersize=12)
ax.set_xlabel("the constant being scored (minutes)")
ax.set_ylabel("MAPE (%)")
ax.set_xlim(10, 50)
ax.set_ylim(0, 62)
ax.set_title("MAPE's best constant sits below the median (the ticks are the nine actuals)",
             fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**MAPE picks 25.00, five minutes below the median of 30.** This confirms 05-01's finding from the other
direction: each metric has its own best constant, and they disagree.

**Why 25 and not 30?** MAPE divides each error by its own actual, so a miss on the 16-minute delivery
counts 45/16 = 2.8 times as much as the same miss on the 45-minute one. The small actuals therefore have
the loudest vote, and they pull the constant down towards themselves. (Formally, the MAPE-optimal
constant is the weighted median with weights `1/actual`.)

**And that is the bias, made concrete.** Nobody asked this model to under-predict; optimising MAPE did it
on its own. Scored with MAE, MAPE's choice is worse - 8.11 minutes against the median's 7.56.

### E11 · Systematically 10% high against systematically 10% low

In [ ]:
ten_high = actual * 1.1
ten_low = actual * 0.9

comparison = pd.DataFrame([
    {"model": "10% high", "MAE": mean_absolute(actual, ten_high),
     "MAPE": percentage(actual, ten_high), "sMAPE": symmetric_percentage(actual, ten_high)},
    {"model": "10% low", "MAE": mean_absolute(actual, ten_low),
     "MAPE": percentage(actual, ten_low), "sMAPE": symmetric_percentage(actual, ten_low)},
])
print(comparison.round(6).to_string(index=False))

**MAE and MAPE cannot tell them apart. sMAPE can - and it prefers the high one.**

MAE is identical because the misses are the same sizes with opposite signs, and MAE throws the sign away.
MAPE is identical for a subtler reason: it divides by the actual, and 10% high means the error *is* 10%
of the actual, in either direction. So MAPE returns exactly 0.1000 both times.

**This is worth pausing on, because it looks like it contradicts the chapter.** It does not. MAPE's
asymmetry lives on the *ratio* scale: predicting 2x the actual costs 100% while predicting half costs
50%. "10% high" and "10% low" are ratios 1.1 and 0.9, which are equally far from 1 in the additive sense,
so MAPE scores them the same. Ask instead for "double" against "half" and the asymmetry appears
immediately.

**sMAPE, meanwhile, has its own bias in the opposite direction:** 9.52% for the high model against 10.53%
for the low one, because its denominator grows when the prediction is high, which shrinks the fraction.
The metric built to fix MAPE's asymmetry simply reversed it.

**The general lesson:** any metric that divides by something the model can influence has a direction it
quietly prefers. MAE divides by nothing, which is a large part of why it is the safe default.

### E12 · Finding where fresh-row R-squared turns negative

In [ ]:
fit_rng = np.random.default_rng(2)
fit_x = fit_rng.uniform(-3, 3, 25)
fit_y = 0.8 * fit_x + fit_rng.normal(0, 1.0, 25)
fresh_x = fit_rng.uniform(-3, 3, 200)
fresh_y = 0.8 * fresh_x + fit_rng.normal(0, 1.0, 200)


def r2_by_rows(model, expand, X_fit, y_fit, X_fresh, y_fresh):
    fitted_r2 = r2_score(y_fit, model.predict(expand.transform(X_fit.reshape(-1, 1))))
    fresh_r2 = r2_score(y_fresh, model.predict(expand.transform(X_fresh.reshape(-1, 1))))
    return float(fitted_r2), float(fresh_r2)


sweep = []
for degree in range(1, 19):
    expand = PolynomialFeatures(degree, include_bias=False)
    model = LinearRegression().fit(expand.fit_transform(fit_x.reshape(-1, 1)), fit_y)
    on_fit, on_fresh = r2_by_rows(model, expand, fit_x, fit_y, fresh_x, fresh_y)
    sweep.append({"degree": degree, "R2 fitted": round(on_fit, 4), "R2 fresh": round(on_fresh, 4)})

sweep = pd.DataFrame(sweep)
print(sweep.to_string(index=False))
first_negative = int(sweep.loc[sweep["R2 fresh"] < 0, "degree"].iloc[0])
print("\nfresh-row R-squared first goes negative at degree %d" % first_negative)

**Degree 7 is where it first goes negative**, at -1.2182: from that point on the model is worse than
predicting the mean on rows it has not seen.

Two things in this table are worth naming.

**The crossing is not gentle.** Degree 6 still scores +0.2008 and degree 7 scores -1.2182. There is no
warning band; one extra column takes the model from mediocre to worse-than-a-constant.

**The fitted column is not monotonic either** - it peaks at 0.8459 (degree 11) and drifts down to 0.8048.
As the chapter noted, that is the arithmetic failing rather than the theory: `x` reaches 3, so the
degree-18 column reaches 3^18 and the least-squares solve becomes numerically unstable.

## Interpretation

### E13 · "MAPE 12%, down from 15%"

Two ways that happens with no improvement in the forecasts:

**1. The actuals got bigger.** MAPE divides by the actual. A quarter with higher volumes produces a lower
MAPE from *identical* absolute errors - the denominators simply grew. Check whether the mean actual rose.

**2. The mix of series changed.** If the small-volume products - the ones with the huge percentage errors
- were discontinued, reported late, or moved to another team's report, MAPE falls because the worst
denominators left the average. Nothing was forecast better.

A third, less innocent version: someone switched from MAPE to weighted APE, or started excluding rows
with actuals below a threshold, and the report kept the same label.

**What to ask for:** the MAE alongside it, on a fixed set of series. If MAE fell too, something real
happened.

### E14 · RMSE 4.2, MAE 1.1, on 500 rows

The ratio is 3.8, far above the ~1.25 of well-behaved errors, and the ceiling for 500 rows is
sqrt(500) = 22.4, so this is well up that range.

**Conclusion: the typical row is fine and a small number of rows are very badly wrong.** The model is not
uniformly mediocre - it is good most of the time with a failure mode.

**First thing to look at: sort the residuals by absolute size and read the top twenty rows.** Not the
metric - the actual rows. What you are looking for is whether they have something in common: one
customer, one product, one date range, one sensor, one category the model never saw enough of. That is a
fixable problem, and no summary metric can tell you which one it is.

## Debugging

### E15 · MAPE is `inf`

**Cause: at least one actual is exactly zero**, so its absolute percentage error divides by zero. A
single such row makes the whole average infinite regardless of the other 10,000.

Two things you *could* do:

1. **Drop the zero-actual rows.** Fast, and it silently deletes the rows the model most needs to be
   judged on. It also makes the number incomparable with last month's.
2. **Switch to weighted APE** - total absolute error over total actual. It never divides by an individual
   row, so zeros cost nothing special, and it still reports a percentage the business can read.

**I would choose the second.** The `inf` is not a bug to be silenced; it is MAPE telling you it is the
wrong metric for this data. Deleting rows to keep a broken metric is fixing the evidence rather than the
measurement. Report weighted APE, or MAE with the mean actual next to it.

### E16 · 0.92 training, 0.10 test

**The more likely explanation is overfitting, not a bad test set.** The chapter's degree sweep is exactly
this pattern: R-squared climbing on the fitting rows while it collapses on fresh ones. Note also that
0.92 was measured where the model was fitted, which 05-03's E9 already showed can be pushed up with pure
noise columns.

**The check - and it is one line: shuffle the split and rerun several times.**

- If the gap follows the *model* across every reshuffle, it is overfitting.
- If the gap only appears with this one split, the split is the problem - and 04-03 gave the reason
  (small test sets are lotteries) while 04-04 gave the case where it is real (a group or a time boundary
  the random split ignored).

A second check worth thirty seconds: how many features and how many training rows? If the counts are
close, the answer is already in front of you.

## Exam and interview reasoning

### E17 · "Which metric would you use?"

> "MAE as the default, because it is in the units of the target and a stakeholder can act on it, and I
> report it next to a baseline so the number means something. I compute RMSE alongside it - not mainly to
> report, but because the RMSE-to-MAE ratio tells me for free whether a few rows are carrying the error.
> If the loss the business actually faces is asymmetric or grows faster than linearly, I say so and match
> the metric to it rather than defaulting."

**"The business always asks for MAPE - what do you tell them?"**

Do not refuse it. Give them the percentage they want in a form that does not break:

> "I can give you a percentage. I would rather it were total error over total volume than the average of
> per-row percentages, because the per-row version is dominated by the smallest orders - one 2-unit order
> can move the headline more than a hundred large ones. Same units, same interpretation, and it does not
> jump when a small product has a quiet week. I will show both this month so you can see the difference
> on our own data."

The move being demonstrated is not "MAPE is bad" - it is that you understood *why* they want a
percentage (comparability across scales) and supplied that requirement with something that works.

## Transfer to a different situation

### E18 · Daily electricity demand

**Optimise: MSE (equivalently RMSE).** Extreme days matter enormously, and squared error is the loss that
takes them seriously - a 500 MW miss counts 25 times a 100 MW miss, which is roughly how the physical
consequences scale. Optimising MAE would let the model be comfortable about the days that matter most.

**Report: MAE in megawatt-hours, with the RMSE beside it.** The operations team needs "we are typically
600 MWh out", not a squared quantity.

**Why they differ:** the optimiser needs the loss whose shape matches the real cost; the reader needs a
number in their own units. Nothing requires these to be the same function, and treating them as one is
the most common metric mistake.

**Two things the question is really testing.**

MAPE is defensible here - demand is never near zero, so its explosion failure does not apply - but it is
still the wrong choice, because its asymmetry pushes forecasts *low* and low forecasts are the direction
that causes blackouts. The metric's bias points at the expensive failure.

Which brings the last piece: **the cost is asymmetric and none of these metrics are.** MAE, RMSE and MAPE
all treat a 100 MW over-forecast the same as a 100 MW under-forecast, and the problem says they are not
the same at all. The honest answer names a **pinball (quantile) loss** - or plainly, that you would
forecast a high quantile rather than the mean, and hold reserve accordingly. Getting to "my metric is
symmetric and my costs are not" is the answer; the machinery has a name and 05-05 onwards will meet it.

## Explain it to someone non-technical

### E19 · "Our forecast is 95% accurate"

> Ninety-five percent of what? If we deliver 400 parcels a day and we are 20 out, that is 95% - and if we
> deliver 20 parcels and we are 1 out, that is also 95%. The two situations need completely different
> staffing. A percentage also hides who was wrong: being 5% off on every single day and being perfect for
> a month then catastrophically wrong on Black Friday both average to 95%, and only one of them costs us
> anything. Tell me instead: how many parcels out are we on a typical day, and what is the worst day?

*(88 words.)* The two things being smuggled back in are **units** and **spread** - which is exactly what
this chapter argued a single percentage throws away.

## Optional challenge

### E20 · The bounds on RMSE / MAE

In [ ]:
n = 9
equal_errors = np.full(n, 2.0)
one_error = np.zeros(n)
one_error[0] = 1.0


def ratio_of(errors):
    return float(np.sqrt((errors ** 2).mean()) / np.abs(errors).mean())


print("all nine errors equal        ratio %.4f   (the lower bound, 1)" % ratio_of(equal_errors))
print("one non-zero error of nine   ratio %.4f   (the upper bound, sqrt(9) = %.4f)"
      % (ratio_of(one_error), np.sqrt(n)))

**The lower bound.** RMSE = MAE requires the mean of the squares to equal the square of the mean, and
Jensen's inequality is an equality only when the quantity being averaged is constant. So the ratio is 1
exactly when every `|error|` is the same - and the size does not matter, only that they agree.

**The upper bound.** With one error of size `e` and the rest zero:
`MAE = e/n` and `RMSE = sqrt(e²/n) = e/sqrt(n)`, so the ratio is `(e/sqrt(n)) / (e/n) = sqrt(n)`.
For n = 9 that is 3, which the code confirms.

Both bounds have a plain-language reading. **The ratio is asking what fraction of the errors are doing
the work.** All of them equally: 1. One of them alone: sqrt(n).

In [ ]:
sizes = [10, 100, 1000, 10000, 100000]
converge_rng = np.random.default_rng(7)
observed = [float(np.mean([ratio_of(e) for e in converge_rng.normal(0, 1, (200, n))]))
            for n in sizes]

for n, value in zip(sizes, observed):
    print("n = %6d   mean ratio %.4f" % (n, value))
print("\nsqrt(pi / 2) = %.4f" % np.sqrt(np.pi / 2))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.semilogx(sizes, observed, "o-", color="#0072B2", linewidth=2.2, markersize=9,
            label="observed mean ratio")
ax.axhline(np.sqrt(np.pi / 2), color="#D55E00", linestyle="--", linewidth=2,
           label="sqrt(pi/2) = 1.2533")
ax.set_xlabel("number of errors (log scale)")
ax.set_ylabel("RMSE / MAE")
ax.set_ylim(1.235, 1.258)
ax.set_title("Normally distributed errors converge to sqrt(pi/2), not to the sqrt(n) ceiling",
             fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**It converges to sqrt(pi/2) = 1.2533.**

For a normal distribution with scale `sigma`, the root mean square of the errors tends to `sigma` and the
mean absolute value tends to `sigma * sqrt(2/pi)`. The ratio is `sqrt(pi/2)`, and `sigma` cancels - so the
constant is a property of the *shape* of the error distribution, not of its size.

**The important part is what does not happen.** The ceiling sqrt(n) grows without limit - at n = 100,000
it is 316 - and the observed ratio does not move towards it at all. From n = 100 onwards every
simulated value is within 0.001 of 1.2533.

So the ratio is not measuring how many errors there are. **It is measuring how unequal they are**, and it
has a known value for well-behaved errors. That is what makes "1.25, fine - 3.8, go and look" a rule you
can carry into any regression problem, at any scale.